In [1]:
import tensorflow as tf

path_to_file = tf.keras.utils.get_file(
    'sherlock_holmes.txt',
    'https://www.gutenberg.org/files/1661/1661-0.txt'
)

# Read the text
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f"Length of text: {len(text)} characters")
print(text[:200]) # Print first 200 chars to verify

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
tokenizer=Tokenizer()

In [ ]:
tokenizer.fit_on_texts([text])

In [ ]:
len(tokenizer.word_index)

In [ ]:
input_sequences = []
for sentence in text.split('\n'):
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1,len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

In [ ]:
input_sequences

In [ ]:
max_len = max([len(x) for x in input_sequences])

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_len, padding='pre')

In [ ]:
padded_input_sequences

In [ ]:
X = padded_input_sequences[:,:-1]

In [ ]:
y = padded_input_sequences[:,-1]

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(y, num_classes=len(tokenizer.word_index) + 1)

In [ ]:
y.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
vocab_size = len(tokenizer.word_index) + 1  # Calculate this once

model = Sequential()

# 1. Embedding Layer
model.add(Embedding(vocab_size, 100, input_length=max_len-1))

# 2. First LSTM Layer
# FIX: Must have return_sequences=True to feed into the next LSTM
model.add(LSTM(200, return_sequences=True))

# 3. Second LSTM Layer
# This is the last LSTM, so it does NOT need return_sequences=True
model.add(LSTM(200))

# 4. Output Layer
# FIX: The number of neurons MUST match vocab_size (not 450)
model.add(Dense(vocab_size, activation='softmax'))


In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# --- 4. TRAIN ---
# 50 epochs should take 2-5 mins
model.fit(X, y, epochs=50, verbose=1)

In [ ]:
import time
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

text = "what is the fee"

# --- SPEED OPTIMIZATION ---
# Create a reverse dictionary once so we don't have to loop every time
reverse_word_map = {index: word for word, index in tokenizer.word_index.items()}

# Ensure this matches your model's expected input length!
# If you trained with max_sequence_len, use (max_sequence_len - 1)
model_input_length = 56

print(f"Start: {text}")

for i in range(10):
    # 1. Tokenize
    token_text = tokenizer.texts_to_sequences([text])[0]

    # 2. Padding
    # usage: [token_text] ensures it is a 2D array (batch of 1)
    padded_token_text = pad_sequences([token_text], maxlen=model_input_length, padding='pre')

    # 3. Predict
    # verbose=0 stops the progress bar spam
    prediction = model.predict(padded_token_text, verbose=0)
    pos = np.argmax(prediction)

    # 4. Get Word (Instant lookup)
    word = reverse_word_map.get(pos)

    if word:
        text = text + " " + word
        print(text)
    else:
        # If the model predicts '0' (padding) or a word not in map
        print("--- End of prediction ---")
        break

    time.sleep(2)